# Week 7 lab: the same questions in Spark and DuckDB

Cascade Mobility asks: *our pipeline handles two months easily. What happens
with a full year, and do we need a "big data" engine like Spark?*

In this notebook you will:

1. Start Spark inside this container and read a full year of silver Parquet (about 6.3 million trips).
2. Answer **three questions**: the same three that `spark/week07_duckdb.py` answers with DuckDB.
3. Write a result back out as **partitioned Parquet**.
4. Time everything, so you can compare the two engines in `WEEK_07.md`.

**Before you start:** the full-year silver data must exist (`uv run python spark/get_full_year.py`),
and the container must be running (`docker compose -f docker/spark.yml up`).
Run the cells from top to bottom with **Shift+Enter**.

## 1. Start Spark

Spark is built to spread work across many machines. Here it runs in **local mode**:
one machine (this container) pretends to be a small cluster, with one *executor
thread* per CPU core.

The container has a hard 3 GB memory limit (set in `docker/spark.yml`), so we give
Spark's driver 1.5 GB and leave the rest for Python and the operating system.

In [ ]:
import time
from pyspark.sql import SparkSession, functions as F

spark = (
    SparkSession.builder
    .appName("cascade-week07")
    .master("local[*]")                                   # use every CPU core in the container
    .config("spark.driver.memory", "1536m")                # stay well under the 3 GB container limit
    .config("spark.sql.shuffle.partitions", "8")          # see "shuffle" below; the default of 200 is for big clusters
    .config("spark.sql.session.timeZone", "America/Chicago")
    .config("spark.sql.sources.partitionColumnTypeInference.enabled", "false")  # keep year_month as text
    .getOrCreate()
)

timings = {}                         # name -> seconds

def timed(name, action):
    """Run action(), remember how long it took, and return its result."""
    start = time.perf_counter()
    result = action()
    timings[name] = time.perf_counter() - start
    print(f"{name}: {timings[name]:.2f} s")
    return result

print("Spark", spark.version, "| cores:", spark.sparkContext.defaultParallelism)

## 2. Read silver

`spark.read.parquet` does **not** read any data yet. Spark is *lazy*: it only
builds a plan. Work happens when you ask for a result (`count()`, `show()`,
`toPandas()`, `write`). That's why we time the *actions*, not the reads.

The folder has one sub-folder per month (`year_month=2026-09`, ...). Spark turns
that folder name back into a `year_month` column, just as DuckDB did in Week 3.

In [ ]:
SILVER = "/home/jovyan/data/silver/trips"

trips = spark.read.parquet(SILVER)
trips.printSchema()

total = timed("count_all_rows", lambda: trips.count())
print(f"{total:,} trips in silver")

### What is a partition?

Spark splits a dataset into **partitions**: chunks of rows that can be processed
independently, one per core at a time. More partitions means more work can run
in parallel, but each one carries a little overhead.

There are two kinds of "partition" in this lab, and they are easy to mix up:

* **Spark partitions**: chunks of rows in memory while a job runs (the number below).
* **Partitioned folders on disk**: `year_month=2026-09/...`. When a query filters
  on `year_month`, Spark (and DuckDB) can skip whole folders without opening them.

In [ ]:
print("Spark partitions while reading silver:", trips.rdd.getNumPartitions())

valid = trips.filter(F.col("is_valid"))      # same rule as gold: only valid trips

## 3. Question 1: trips per month by member type

In [ ]:
q1 = (
    valid.groupBy("year_month", "member_casual")
         .agg(F.count("*").alias("trips"))
         .orderBy("year_month", "member_casual")
)
timed("q1_trips_by_month_and_member", lambda: q1.toPandas())

## 4. Question 2: top 10 start stations, and what a *shuffle* is

To count trips per station, every row for "Navy Pier" has to end up in the same
place. But Navy Pier's rows are spread across all partitions. So Spark:

1. counts within each partition (cheap, no data moves),
2. **shuffles**: sends each partial count to the partition responsible for that
   station name (data moves between cores, or between machines on a real cluster),
3. adds the partial counts together.

The shuffle is usually the most expensive step in a Spark job. In the plan below,
look for **`Exchange hashpartitioning(start_station_name, ...)`**: that is the shuffle.
`spark.sql.shuffle.partitions = 8` (set above) controls how many partitions come
out of it.

In [ ]:
# TODO(week07): Question 2. Count VALID trips per start_station_name (leave out
# trips with no station name), keep the 10 busiest, busiest first (ties: by name).
# Use groupBy / agg / orderBy / limit, like Question 1.
q2 = None   # your DataFrame here

q2.explain()      # read the plan from the bottom up: find the Exchange (the shuffle)
timed("q2_top10_start_stations", lambda: q2.toPandas())

## 5. Question 3: average trip length by hour and bike type

In [ ]:
# TODO(week07): Question 3. For VALID trips: by start hour (0-23) and rideable_type,
# the number of trips and the average duration_min rounded to 2 decimals.
# Columns: start_hour, rideable_type, trips, avg_minutes. Order by hour, then bike type.
q3 = None   # your DataFrame here

timed("q3_avg_minutes_by_hour_and_bike", lambda: q3.toPandas())

## 6. Write partitioned Parquet

We write *trips per start station per day*, partitioned by `year_month`. Each
month becomes its own folder, just like silver.

`mode("overwrite")` replaces the whole output on every run. That makes this step
**idempotent**: run the cell twice and you get the same files, not double the rows.

In [ ]:
OUT = "/home/jovyan/data/engine_out/spark/station_daily"

# TODO(week07): trips per start station per day, for VALID trips with a start_station_id.
# Columns: start_station_id, trip_date (the date of started_at), year_month, trips.
station_daily = None   # your DataFrame here

# Write it partitioned by year_month, replacing any earlier output.
timed("write_station_daily",
      lambda: station_daily.write.mode("overwrite").partitionBy("year_month").parquet(OUT))

written = spark.read.parquet(OUT)
print(f"{written.count():,} station-day rows in", written.select("year_month").distinct().count(), "month folders")

## 7. Your timings

Copy this table into `docs/WEEK_07.md` (use `docs/week07_comparison_template.md`),
then run the DuckDB version on your laptop and fill in the other column:

```bash
uv run python spark/week07_duckdb.py
```

The first Spark action is slower because the Java virtual machine is still
warming up. Try running a question cell a second time and see what changes.

In [ ]:
for name, seconds in timings.items():
    print(f"{name:36s} {seconds:7.2f} s")

In [ ]:
spark.stop()   # frees the memory; restart the kernel to run again